# Сборка аналитического датасета и EDA: ФП → Дефолт

**Контрольная точка:** КТ-3 (Результаты разведочного анализа)  
**Этап:** Шаг 3. Разведочный анализ и визуализация

---

### Цель ноутбука

Собрать единый аналитический датасет из трёх источников и провести разведочный анализ (EDA), чтобы ответить на вопрос: **какие факторы проблемности (ФП) связаны с дефолтом компании?**

### Единица наблюдения

**Компания (ИНН)**. Каждая строка итогового датасета — одна уникальная компания.

### Источники данных

| Источник | Роль | Таблица в Озере | Локальный файл | Строк | Колонок | Период |
|----------|------|----------------|----------------|------:|--------:|--------|
| **CRM** | **Основной** (ФП + вселенная) | `sandbox_ai.tmp_ecp_crm_fp_svy` | `data_crm.csv` | 1 504 263 | 41 | 2022-04 — 2025-12 |
| **H2.0** | Вспомогательный (сегмент) | `sandbox_ai.tmp_h20_fp_svy` | `data_h20.xlsx` | 172 981 | 11 | 2023-02 — 2025-12 |
| **Дефолты** | **Основной** (целевая переменная) | `sandbox_ai.tmp_test_default_svy` | `data_defaults.pkl` | 889 | 11 | 2020-01 — 2025-12 |

### Ключевые решения по архитектуре

1. **ФП берём из CRM** — колонка `NUMBER_FP_SFP` (тип ФП, 100% заполненность). Дата выявления — `IDENTIFICATION_DATE`. Источники: Н2.0, Справочник CRM-системы, CRM-система.
2. **Реестр компаний** берём из CRM (`X_INN`). Компании без ФП в датасет не попадают.
3. **Временной фильтр**: ФП считаются только если выявлены **до** дефолта (или до отсечки для недефолтных) — исключает data leakage.
4. **Все компании** в выгрузке дефолтов получают `default_flag = 1`, тяжесть классифицируется отдельно.

### Структура итогового датасета

| Колонка | Тип | Описание |
|---------|-----|----------|
| `inn` | str | ИНН компании (ключ) |
| `default_flag` | int8 | 1 = дефолт, 0 = нет |
| `default_date` | datetime | Дата первого дефолта (NaT для недефолтных) |
| `severity` | str | Тяжесть: тяжёлый / активный / надзор / вышел из дефолта / нет дефолта |
| `fp_*` | int8 | N бинарных колонок по числу уникальных NUMBER_FP_SFP: 1 = ФП был, 0 = нет |

### Разделы EDA (deliverables КТ-3)

| № | Раздел | Что выдаёт |
|---|--------|------------|
| 7 | Гистограммы ФП | Распределение числа ФП у дефолтных vs недефолтных |
| 8 | Динамика дефолтов | По месяцам и кварталам |
| 9 | Тепловая карта | Корреляции между ТОП-30 ФП |
| 10 | Кросс-таблицы | Частота, доля дефолтов, Lift по каждому ФП |
| 10.1 | Комбинации | ТОП-5 пар и троек ФП по Lift |
| 11 | Аномалии | ФП с 100%/0% дефолтностью, выбросы, дефолты без ФП |
| 12 | Гипотезы | Направления для углублённого анализа (шаги 4–6) |

## 0. Импорты и настройки

In [ ]:
import pandas as pd
import numpy as np
import warnings

# Подавляем FutureWarning от pandas и DeprecationWarning от openpyxl,
# чтобы вывод ноутбука оставался чистым для заказчика
warnings.filterwarnings("ignore")

# Расширяем отображение таблиц в Jupyter, чтобы не терять колонки/строки
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

## 1. Загрузка данных

In [ ]:
DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"

SEGMENT_MAP = {
    "ДМСБ (микро)":   "МкБ",
    "ДМСБ (малый)":   "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ":            "ККБ",
}
SEG_ORDER = ["МкБ", "МСБ", "ККБ"]

ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]


def normalize_inn(s):
    """Приводит ИНН к единому строковому формату (10 или 12 знаков).
    Убирает '.0' от Excel, ведущие нули, дополняет до стандартной длины."""
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)

# --- CRM: основной источник ФП ---
df_crm = pd.read_csv(f"{DATA_DIR}/crm_last_version.csv", sep=";", encoding="utf-8-sig", dtype=str, low_memory=False)
df_crm.columns = df_crm.columns.str.strip()
df_crm = df_crm.rename(columns={
    'VAL.1': 'VAL_1', 'DESC_TEXT.1': 'DESC_TEXT_1', 'ROW_ID.1': 'ROW_ID_1',
    'AGREEMENT_NUM.1': 'AGREEMENT_NUM_1', 'ROW_ID.2': 'ROW_ID_2',
    'VAL.2': 'VAL_2', 'NAME.1': 'NAME_1', 'VAL.3': 'VAL_3', 'VAL.5': 'VAL_5',
})
print(f"CRM:      {df_crm.shape[0]:>10,} строк × {df_crm.shape[1]} колонок")

# --- H2O: вспомогательный источник ---
df_h2o = pd.read_excel(f"{DATA_DIR}/2023-2025_h20_data.xlsx", dtype=str)
df_h2o.columns = df_h2o.columns.str.strip()
df_h2o = df_h2o.rename(columns={
    'ИНН': 'inn', 'Наименование клиента': 'client_name',
    'CDI ID клиента': 'cdi_id', 'ОПФ': 'legal_form', 'РФ': 'rf',
    'Сегмент (зона ответственности)': 'segment', 'Вид мониторинга': 'mon_type',
    'ИД фактора на клиенте в CRM': 'crm_fp_id',
    'ИД справочного фактора в CRM': 'h20_fp_id',
    'Номер фактора': 'ref_book_fp_id',
    'Дата выявления фактора проблемности': 'fp_start_date',
})
df_h2o["inn"] = df_h2o["inn"].apply(normalize_inn)
print(f"H2O:      {df_h2o.shape[0]:>10,} строк × {df_h2o.shape[1]} колонок")

# --- Дефолты: история дефолтов компаний ---
df_def = pd.read_pickle(f"{DATA_DIR}/default_data.pkl")
df_def = df_def.astype(str).replace("nan", np.nan)
df_def.columns = df_def.columns.str.strip()
df_def = df_def.rename(columns={'start': 'start_date', 'cure': 'cure_date', 'finish': 'finish_date'})
df_def["inn"] = df_def["inn"].apply(normalize_inn)
print(f"Дефолты:  {df_def.shape[0]:>10,} строк × {df_def.shape[1]} колонок")

## 2. Подготовка таблицы дефолтов

Все компании в выгрузке defaults подверглись дефолту. Тяжесть:
- **unlimited_default=1** → тяжёлый дефолт (бессрочный, присваивается сразу, без дальнейших проверок)
- **writeoff или liquidation заполнены** → тяжёлый дефолт (списание или ликвидация — это даты, заполняется одно из двух)
- **finish_date заполнена** → компания вышла из дефолта (излечилась)
- **cure_date заполнена, finish_date пуста** → компания на стадии надзора (6 мес)
- Остальные → активный дефолт

`default_flag = 1` для **всех** компаний из этой выгрузки.

In [ ]:
# Приводим колонки дат к datetime
for col in ["start_date", "cure_date", "finish_date", "writeoff", "liquidation"]:
    if col in df_def.columns:
        df_def[col] = pd.to_datetime(df_def[col], dayfirst=True, errors="coerce")

# unlimited_default — единственный флаг 0/1
if "unlimited_default" in df_def.columns:
    df_def["unlimited_default"] = pd.to_numeric(df_def["unlimited_default"], errors="coerce").fillna(0).astype(int)


def classify_severity(row):
    """Классифицирует тяжесть дефолта по приоритету (от худшего к лучшему):
    
    1. «тяжёлый»          — unlimited_default=1 (бессрочный, сразу тяжелейший)
                            или writeoff/liquidation заполнены (списание/ликвидация)
    2. «активный»         — компания в процессе дефолта (нет cure_date, нет finish_date)
    3. «надзор»           — есть cure_date, но finish_date ещё не наступила
    4. «вышел из дефолта» — finish_date заполнена (дефолт закрыт)
    """
    if row.get("unlimited_default") == 1:
        return "тяжёлый"
    if pd.notna(row.get("writeoff")) or pd.notna(row.get("liquidation")):
        return "тяжёлый"
    if pd.notna(row.get("finish_date")):
        return "вышел из дефолта"
    if pd.notna(row.get("cure_date")):
        return "надзор"
    return "активный"


df_def["severity"] = df_def.apply(classify_severity, axis=1)

print(f"Всего строк дефолтов: {len(df_def):,}")
print(f"Уникальных ИНН в дефолтах: {df_def['inn'].nunique():,}")
print(f"Период: {df_def['start_date'].min()} — {df_def['start_date'].max()}")
print(f"\nРаспределение по тяжести:")
print(df_def["severity"].value_counts().to_string())

In [ ]:
# Агрегация до уровня компании: одна строка на ИНН.
# У одной компании может быть несколько записей о дефолте (повторные эпизоды).
# Логика: берём наихудшую тяжесть и самую раннюю дату начала дефолта.

# Приоритет тяжести: 0 = худший (тяжёлый), 3 = лучший (вышел)
severity_priority = {"тяжёлый": 0, "активный": 1, "надзор": 2, "вышел из дефолта": 3}
df_def["_sev_rank"] = df_def["severity"].map(severity_priority)

# Фильтруем по тому же периоду, что и CRM/H2O
date_from = pd.Timestamp("2023-02-01")
date_to   = pd.Timestamp("2025-12-30")

defaults = (
    df_def
    .dropna(subset=["inn", "start_date"])
    .query("@date_from <= start_date <= @date_to")
    .sort_values("_sev_rank")
    .groupby("inn", as_index=False)
    .agg(
        default_date=("start_date", "min"),      # самая ранняя дата дефолта
        severity=("severity", "first"),           # наихудшая тяжесть (first после sort)
        writeoff=("writeoff", "min"),             # самая ранняя дата списания (NaT если не было)
        liquidation=("liquidation", "min"),       # самая ранняя дата ликвидации (NaT если не было)
        finish_date=("finish_date", "min"),       # самая ранняя дата закрытия дефолта
        cure_date=("cure_date", "min"),           # самая ранняя дата выхода на надзор
        unlimited_default=("unlimited_default", "max"),  # 1, если есть бессрочный
    )
)
defaults["default_flag"] = 1  # все компании из этой таблицы — дефолтные

print(f"Уникальных компаний-дефолтников: {len(defaults):,}")
print(f"Период дефолтов: {defaults['default_date'].min()} — {defaults['default_date'].max()}")
print(f"\nТяжесть (на уровне компании):")
print(defaults["severity"].value_counts().to_string())

## 3. Подготовка таблицы ФП из CRM

`NUMBER_FP_SFP` — тип ФП (100% заполненность). `IDENTIFICATION_DATE` — дата выявления ФП.

CRM — мастер-система, содержит все ФП. Фильтруем по источникам (Н2.0, Справочник CRM-системы, CRM-система) и по периоду 2023-02-01 — 2025-12-30.

In [ ]:
# Конвертируем дату выявления ФП в datetime
df_crm["IDENTIFICATION_DATE"] = pd.to_datetime(
    df_crm["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce"
)

# Нормализация ИНН (единый подход для всех тетрадок)
df_crm["X_INN"] = df_crm["X_INN"].apply(normalize_inn)

# Маппинг сегментов и исключение немаппированных
df_crm["X_AREA_RESP"] = df_crm["X_AREA_RESP"].astype(str).str.strip()
df_crm["_segment"] = df_crm["X_AREA_RESP"].map(SEGMENT_MAP)
_unmapped = df_crm[df_crm["_segment"].isna()]["X_AREA_RESP"].value_counts()
if len(_unmapped) > 0:
    print(f"Немаппированные X_AREA_RESP (исключены):")
    print(_unmapped.to_string())
df_crm = df_crm[df_crm["_segment"].notna()].copy()
df_crm = df_crm.drop(columns=["_segment"])
print(f"После маппинга сегментов: {len(df_crm):,} строк")

# Фильтруем по источникам: только Н2.0, Справочник CRM-системы, CRM-система
df_crm = df_crm[df_crm["VAL"].str.strip().isin(ALLOWED_SOURCES)].copy()
print(f"После фильтра по источникам ({', '.join(ALLOWED_SOURCES)}): {len(df_crm):,} строк")

# Фильтруем по периоду 2023-02-01 — 2025-12-30 (согласованность с H2O)
date_from = pd.Timestamp("2023-02-01")
date_to   = pd.Timestamp("2025-12-30")
df_crm = df_crm[
    (df_crm["IDENTIFICATION_DATE"] >= date_from) &
    (df_crm["IDENTIFICATION_DATE"] <= date_to)
].copy()

# Формируем таблицу ФП из CRM:
#   X_INN               → inn       (ИНН компании)
#   NUMBER_FP_SFP       → fp_type   (тип ФП, 100% заполнен)
#   IDENTIFICATION_DATE → fp_date   (дата выявления)
fp = df_crm[["X_INN", "NUMBER_FP_SFP", "IDENTIFICATION_DATE"]].copy()
fp = fp.rename(columns={
    "X_INN": "inn",
    "NUMBER_FP_SFP": "fp_type",
    "IDENTIFICATION_DATE": "fp_date",
})

fp = fp.dropna(subset=["inn", "fp_type"])

print(f"Строк ФП (CRM, после фильтра дат): {len(fp):,}")
print(f"Уникальных типов ФП (NUMBER_FP_SFP): {fp['fp_type'].nunique()}")
print(f"Уникальных ИНН в CRM: {fp['inn'].nunique():,}")
print(f"Период ФП: {fp['fp_date'].min()} — {fp['fp_date'].max()}")

# --- Сегмент компании из CRM (X_AREA_RESP → SEGMENT_MAP) ---
_seg = df_crm[["X_INN", "X_AREA_RESP"]].dropna(subset=["X_INN"]).drop_duplicates()
_mode = lambda s: s.mode().iloc[0] if len(s.mode()) > 0 else None
seg_company = (
    _seg.groupby("X_INN")
    .agg(segment=("X_AREA_RESP", _mode))
    .reset_index()
    .rename(columns={"X_INN": "inn"})
)
seg_company["segment"] = seg_company["segment"].str.strip().map(SEGMENT_MAP)

print(f"\nСегменты компаний:")
print(seg_company["segment"].value_counts(dropna=False).to_string())

## 4. Реестр компаний

Реестр берём из **CRM** (мастер-система). Присоединяем флаг и тяжесть дефолта.

In [ ]:
# Уникальные ИНН из CRM (мастер-система)
companies = fp[["inn"]].drop_duplicates().copy()

# LEFT JOIN с дефолтами
companies = companies.merge(
    defaults[["inn", "default_flag", "default_date", "severity"]],
    on="inn", how="left"
)
companies["default_flag"] = companies["default_flag"].fillna(0).astype(int)
companies["severity"] = companies["severity"].fillna("нет дефолта")

# LEFT JOIN с сегментами
companies = companies.merge(seg_company[["inn", "segment"]], on="inn", how="left")

print(f"Всего уникальных компаний: {len(companies):,}")
print(f"Из них дефолтных:          {companies['default_flag'].sum():,}")
print(f"Default rate:              {companies['default_flag'].mean():.2%}")
print(f"\nРаспределение по severity:")
print(companies["severity"].value_counts().to_string())
print(f"\nРаспределение по сегментам:")
print(companies["segment"].value_counts(dropna=False).to_string())
print(f"\nDefault rate по сегментам:")
for seg in SEG_ORDER:
    sub = companies[companies["segment"] == seg]
    if len(sub) > 0:
        print(f"  {seg}: {sub['default_flag'].mean():.2%} ({int(sub['default_flag'].sum()):,} / {len(sub):,})")

## 5. Временной фильтр ФП

**Критично для корректности:**
- Для дефолтных: оставляем ФП, выявленные **до** даты дефолта
- Для недефолтных: оставляем ФП, выявленные **до** даты отсечки

In [ ]:
# Дата отсечки для недефолтных компаний: конец периода наблюдения.
# Для них учитываем все ФП, выявленные до этой даты.
CUTOFF = pd.Timestamp("2025-12-31")

# Присоединяем к каждому ФП информацию о дефолте компании
fp_with_def = fp.merge(
    companies[["inn", "default_flag", "default_date"]], on="inn", how="inner"
)

# Референсная дата: дефолтные → дата дефолта, недефолтные → CUTOFF
fp_with_def["ref_date"] = fp_with_def["default_date"].fillna(CUTOFF)

# ГЛАВНЫЙ ФИЛЬТР: оставляем только ФП, выявленные СТРОГО ДО референсной даты.
# Это исключает data leakage — ФП, появившиеся после дефолта, не могли его «предсказать».
before = len(fp_with_def)
fp_filtered = fp_with_def[fp_with_def["fp_date"] < fp_with_def["ref_date"]].copy()
after = len(fp_filtered)

print(f"ФП до временного фильтра:  {before:,}")
print(f"ФП после временного фильтра: {after:,}")
print(f"Отсечено (ФП после дефолта): {before - after:,}")

## 6. Pivot — широкий формат

In [ ]:
# Вспомогательная колонка для pivot: наличие ФП = 1
fp_filtered["value"] = 1

# Pivot: длинная таблица (1 строка = 1 ФП) → широкая (1 строка = 1 компания).
# Колонки = типы ФП (NUMBER_FP_SFP), значения = 0/1.
# aggfunc="max": если один тип ФП встретился несколько раз — всё равно 1.
fp_wide = (
    fp_filtered
    .pivot_table(index="inn", columns="fp_type", values="value",
                 aggfunc="max", fill_value=0)
    .reset_index()
)
fp_wide.columns.name = None

# Собираем итоговый датасет: fp_wide + информация о дефолте из universe.
# INNER JOIN: в датасет попадают только компании, у которых есть хотя бы один ФП.
meta_cols = ["inn", "default_flag", "default_date", "severity", "segment"]
dataset = fp_wide.merge(companies[meta_cols], on="inn", how="left")
dataset["default_flag"] = dataset["default_flag"].fillna(0).astype("int8")
dataset["severity"] = dataset["severity"].fillna("нет дефолта")

# Определяем колонки ФП (всё, что не мета-данные)
fp_cols = [c for c in dataset.columns if c not in meta_cols]
dataset[fp_cols] = dataset[fp_cols].fillna(0).astype("int8")

n_def_in_companies = int(companies["default_flag"].sum())
n_def_in_dataset = int(dataset["default_flag"].sum())
n_lost = n_def_in_companies - n_def_in_dataset

print(f"Итоговый датасет: {dataset.shape[0]:,} строк × {dataset.shape[1]} колонок")
print(f"  — компаний:     {dataset.shape[0]:,}")
print(f"  — дефолтных:    {n_def_in_dataset:,}")
print(f"  — ФП-колонок:   {len(fp_cols)}")
print(f"  — в памяти:     {dataset.memory_usage(deep=True).sum() / 1e6:.1f} МБ")

if n_lost > 0:
    print(f"\n⚠ В реестре компаний (п.4) было {n_def_in_companies} дефолтных, "
          f"в датасете осталось {n_def_in_dataset}.")
    print(f"  Причина: {n_lost} дефолтных компаний не имеют ни одного ФП, "
          f"выявленного ДО даты дефолта (отсечены временным фильтром в п.5).")

print(f"\nДефолтные по severity:")
print(dataset[dataset["default_flag"]==1]["severity"].value_counts().to_string())

print(f"\nДатасет по сегментам:")
for seg in SEG_ORDER:
    sub = dataset[dataset["segment"] == seg]
    n_d = int(sub["default_flag"].sum())
    print(f"  {seg}: {len(sub):,} компаний, {n_d:,} дефолтных ({n_d/len(sub):.2%} DR)" if len(sub) > 0 else f"  {seg}: 0 компаний")
no_seg = dataset["segment"].isna().sum()
if no_seg > 0:
    print(f"  Без сегмента: {no_seg:,} компаний")

## 7. Гистограммы распределения ФП: дефолтные vs недефолтные

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Считаем общее число ФП на каждую компанию (сумма по бинарным колонкам)
fp_per_company = dataset[fp_cols].sum(axis=1)
dataset["fp_count"] = fp_per_company

def_counts = fp_per_company[dataset["default_flag"] == 1]
nodef_counts = fp_per_company[dataset["default_flag"] == 0]

print("Число ФП на компанию:")
print(f"  {'':20s} {'Дефолтные':>12s} {'Недефолтные':>12s}")
print(f"  {'Среднее':20s} {def_counts.mean():>12.1f} {nodef_counts.mean():>12.1f}")
print(f"  {'Медиана':20s} {def_counts.median():>12.0f} {nodef_counts.median():>12.0f}")
print(f"  {'Макс':20s} {def_counts.max():>12.0f} {nodef_counts.max():>12.0f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Обрезаем по 95-му перцентилю, чтобы выбросы не сжимали основную часть гистограммы
max_fp = int(fp_per_company.quantile(0.95)) + 1
bins = range(0, max_fp + 2)

# Левый график: абсолютные числа (показывает масштаб дисбаланса классов)
axes[0].hist(nodef_counts.clip(upper=max_fp), bins=bins, alpha=0.7, label="Недефолтные", color="steelblue", edgecolor="white")
axes[0].hist(def_counts.clip(upper=max_fp), bins=bins, alpha=0.7, label="Дефолтные", color="tomato", edgecolor="white")
axes[0].set_xlabel("Количество ФП")
axes[0].set_ylabel("Число компаний")
axes[0].set_title("Распределение количества ФП")
axes[0].legend()

# Правый график: плотность (нормализованные, чтобы сравнивать форму распределений)
axes[1].hist(nodef_counts.clip(upper=max_fp), bins=bins, alpha=0.7, density=True, label="Недефолтные", color="steelblue", edgecolor="white")
axes[1].hist(def_counts.clip(upper=max_fp), bins=bins, alpha=0.7, density=True, label="Дефолтные", color="tomato", edgecolor="white")
axes[1].set_xlabel("Количество ФП")
axes[1].set_ylabel("Доля")
axes[1].set_title("Нормализованное распределение (плотность)")
axes[1].legend()

plt.tight_layout()
plt.show()

### 7.1. Гистограммы распределения ФП по сегментам

In [ ]:
seg_stats = []

for seg in SEG_ORDER:
    seg_data = dataset[dataset["segment"] == seg]
    if len(seg_data) == 0:
        continue
    fp_per = seg_data[fp_cols].sum(axis=1)
    d = fp_per[seg_data["default_flag"] == 1]
    nd = fp_per[seg_data["default_flag"] == 0]

    seg_stats.append({
        "Сегмент": seg,
        "Компаний": len(seg_data),
        "Дефолтных": int(seg_data["default_flag"].sum()),
        "DR": f"{seg_data['default_flag'].mean():.2%}",
        "Среднее ФП (деф.)": round(d.mean(), 1) if len(d) > 0 else "-",
        "Среднее ФП (недеф.)": round(nd.mean(), 1) if len(nd) > 0 else "-",
        "Медиана ФП (деф.)": int(d.median()) if len(d) > 0 else "-",
        "Медиана ФП (недеф.)": int(nd.median()) if len(nd) > 0 else "-",
        "Макс ФП (деф.)": int(d.max()) if len(d) > 0 else "-",
        "Макс ФП (недеф.)": int(nd.max()) if len(nd) > 0 else "-",
    })

    max_fp_seg = int(fp_per.quantile(0.95)) + 1
    bins_seg = range(0, max_fp_seg + 2)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].hist(nd.clip(upper=max_fp_seg), bins=bins_seg, alpha=0.7, label="Недефолтные", color="steelblue", edgecolor="white")
    axes[0].hist(d.clip(upper=max_fp_seg), bins=bins_seg, alpha=0.7, label="Дефолтные", color="tomato", edgecolor="white")
    axes[0].set_xlabel("Количество ФП")
    axes[0].set_ylabel("Число компаний")
    axes[0].set_title(f"{seg} — распределение количества ФП")
    axes[0].legend()

    axes[1].hist(nd.clip(upper=max_fp_seg), bins=bins_seg, alpha=0.7, density=True, label="Недефолтные", color="steelblue", edgecolor="white")
    axes[1].hist(d.clip(upper=max_fp_seg), bins=bins_seg, alpha=0.7, density=True, label="Дефолтные", color="tomato", edgecolor="white")
    axes[1].set_xlabel("Количество ФП")
    axes[1].set_ylabel("Доля")
    axes[1].set_title(f"{seg} — нормализованное распределение")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

print("Сводная таблица по сегментам:")
display(pd.DataFrame(seg_stats).style.hide(axis="index"))

## 8. Закрытие дефолтов по месяцам / кварталам

Дата окончания дефолта (`end_date`) определяется по приоритету: `writeoff` → `liquidation` → `finish_date` → `cure_date`.
Если ни одна дата не заполнена — дефолт считается открытым.

Графики показывают количество компаний, у которых `end_date` попадает в 2023–2025, **без привязки к `start_date`**.

In [ ]:
# --- Привязка сегмента к дефолтам ---
df_def = df_def.merge(seg_company[["inn", "segment"]], on="inn", how="left")
defaults = defaults.merge(seg_company[["inn", "segment"]], on="inn", how="left")

# --- Единая дата окончания дефолта (приоритет: writeoff → liquidation → finish_date → cure_date) ---
df_def["end_date"] = (
    df_def["writeoff"]
    .combine_first(df_def["liquidation"])
    .combine_first(df_def["finish_date"])
    .combine_first(df_def["cure_date"])
)
df_def["end_date"] = pd.to_datetime(df_def["end_date"])

defaults["end_date"] = (
    defaults["writeoff"]
    .combine_first(defaults["liquidation"])
    .combine_first(defaults["finish_date"])
    .combine_first(defaults["cure_date"])
)
defaults["end_date"] = pd.to_datetime(defaults["end_date"])

closed_all = df_def[df_def["end_date"].notna()].copy()
closed_all = closed_all[
    (closed_all["end_date"] >= pd.Timestamp("2023-01-01")) &
    (closed_all["end_date"] <= pd.Timestamp("2025-12-31"))
]
closed_all["end_month"]   = closed_all["end_date"].dt.to_period("M")
closed_all["end_quarter"] = closed_all["end_date"].dt.to_period("Q")

monthly  = closed_all.groupby("end_month")["inn"].nunique()
monthly.index = monthly.index.astype(str)
quarterly = closed_all.groupby("end_quarter")["inn"].nunique()
quarterly.index = quarterly.index.astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].bar(range(len(monthly)), monthly.values, color="indianred", edgecolor="white")
axes[0].set_xticks(range(len(monthly)))
axes[0].set_xticklabels(monthly.index, rotation=45, ha="right", fontsize=8)
axes[0].set_ylabel("Количество компаний")
axes[0].set_title("Закрытие дефолтов по месяцам (за период 2023-2025, с любой датой начала дефолта)")

axes[1].bar(range(len(quarterly)), quarterly.values, color="coral", edgecolor="white")
axes[1].set_xticks(range(len(quarterly)))
axes[1].set_xticklabels(quarterly.index, rotation=45, ha="right")
axes[1].set_ylabel("Количество компаний")
axes[1].set_title("Закрытие дефолтов по кварталам (за период 2023-2025, с любой датой начала дефолта)")

plt.tight_layout()
plt.show()

print(f"Компаний с закрытым дефолтом (end_date в 2023–2025): {closed_all['inn'].nunique()}")
print("\nПо месяцам:")
print(monthly.to_string())
print("\nПо кварталам:")
print(quarterly.to_string())

In [ ]:
# --- Статистика: закрыт / открыт ---
closed = df_def["end_date"].notna().sum()
opened = df_def["end_date"].isna().sum()
print("Статус дефолтов (вся выгрузка df_def):")
print("=" * 55)
print(f"  Дефолт закрыт (есть дата окончания): {closed}  ({closed / len(df_def):.1%})")
print(f"  Дефолт открыт (только start_date):   {opened}  ({opened / len(df_def):.1%})")
print(f"  Всего записей: {len(df_def)}")

closed_p = defaults["end_date"].notna().sum()
opened_p = defaults["end_date"].isna().sum()
print(f"\nСтатус дефолтов за выбранный период (defaults):")
print(f"  Дефолт закрыт: {closed_p}  ({closed_p / len(defaults):.1%})")
print(f"  Дефолт открыт: {opened_p}  ({opened_p / len(defaults):.1%})")
print(f"  Всего записей: {len(defaults)}")

# --- Проверка: ИНН с несколькими различными start_date ---
multi_start = (
    df_def.groupby("inn")["start_date"]
    .nunique()
    .loc[lambda x: x > 1]
    .sort_values(ascending=False)
)
print(f"\n{'=' * 55}")
print(f"ИНН с несколькими различными start_date: {len(multi_start)}")
if len(multi_start) > 0:
    print(f"\nПримеры (до 5 ИНН):")
    for inn in multi_start.head(5).index:
        dates = sorted(df_def.loc[df_def["inn"] == inn, "start_date"].dropna().unique())
        print(f"  ИНН {inn}: {len(dates)} дат → {', '.join(str(d.date()) for d in dates)}")

### 8.1. Закрытие дефолтов по месяцам / кварталам (start_date и end_date в периоде)

Те же закрытые дефолты, но дополнительно фильтруем: и `start_date`, и `end_date` должны попадать в 2023–2025.
Это позволяет увидеть дефолты, которые **целиком** прошли внутри анализируемого периода.

In [ ]:
# end_date уже рассчитана выше; фильтруем: И start_date, И end_date в 2023–2025
closed_both = df_def[
    df_def["end_date"].notna() &
    (df_def["start_date"] >= pd.Timestamp("2023-01-01")) &
    (df_def["start_date"] <= pd.Timestamp("2025-12-31")) &
    (df_def["end_date"]   >= pd.Timestamp("2023-01-01")) &
    (df_def["end_date"]   <= pd.Timestamp("2025-12-31"))
].copy()
closed_both["end_month"]   = closed_both["end_date"].dt.to_period("M")
closed_both["end_quarter"] = closed_both["end_date"].dt.to_period("Q")

monthly_b  = closed_both.groupby("end_month")["inn"].nunique()
monthly_b.index = monthly_b.index.astype(str)
quarterly_b = closed_both.groupby("end_quarter")["inn"].nunique()
quarterly_b.index = quarterly_b.index.astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].bar(range(len(monthly_b)), monthly_b.values, color="indianred", edgecolor="white")
axes[0].set_xticks(range(len(monthly_b)))
axes[0].set_xticklabels(monthly_b.index, rotation=45, ha="right", fontsize=8)
axes[0].set_ylabel("Количество компаний")
axes[0].set_title("Закрытие дефолтов по месяцам (дата начала дефолта находится в диапазоне 2023-2025)")

axes[1].bar(range(len(quarterly_b)), quarterly_b.values, color="coral", edgecolor="white")
axes[1].set_xticks(range(len(quarterly_b)))
axes[1].set_xticklabels(quarterly_b.index, rotation=45, ha="right")
axes[1].set_ylabel("Количество компаний")
axes[1].set_title("Закрытие дефолтов по кварталам (дата начала дефолта находится в диапазоне 2023-2025)")

plt.tight_layout()
plt.show()

print(f"Компаний с закрытым дефолтом (start_date И end_date в периоде): {closed_both['inn'].nunique()}")
print("\nПо месяцам:")
print(monthly_b.to_string())
print("\nПо кварталам:")
print(quarterly_b.to_string())

### 8.2. Среднее время прохождения дефолта (все исторические данные)

Для компаний с закрытым дефолтом (`end_date` определена) рассчитываем длительность = `end_date − start_date` (в днях).
Используется **вся** выгрузка дефолтов (`df_def`), без фильтра по периоду.

In [ ]:
# Длительность закрытых дефолтов (вся выгрузка)
df_closed = df_def[df_def["end_date"].notna()].copy()
df_closed["duration_days"] = (df_closed["end_date"] - df_closed["start_date"]).dt.days
df_closed = df_closed[df_closed["duration_days"] >= 0]

dur = df_closed["duration_days"]

print("Среднее время прохождения дефолта (все исторические данные):")
print("=" * 55)
print(f"  Записей (закрытые дефолты): {len(dur)}")
print(f"  Среднее:   {dur.mean():.0f} дней")
print(f"  Медиана:   {dur.median():.0f} дней")
print(f"  Мин:       {dur.min():.0f} дней")
print(f"  Макс:      {dur.max():.0f} дней")

max_days = int(dur.quantile(0.95)) + 50
bins = range(0, max_days, 30)

fig, ax = plt.subplots(figsize=(14, 5))
ax.hist(dur.clip(upper=max_days), bins=bins, color="steelblue", edgecolor="white")
ax.axvline(dur.mean(), color="tomato", ls="--", lw=2, label=f"Среднее ({dur.mean():.0f} дн.)")
ax.axvline(dur.median(), color="orange", ls="--", lw=2, label=f"Медиана ({dur.median():.0f} дн.)")
ax.set_xlabel("Длительность дефолта (дни)")
ax.set_ylabel("Количество компаний")
ax.set_title("Распределение длительности закрытых дефолтов (все данные)")
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    ["Среднее", "Медиана"],
    [dur.mean(), dur.median()],
    color=["tomato", "steelblue"], edgecolor="white"
)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f"{bar.get_height():.0f}", ha="center", fontsize=11)
ax.set_ylabel("Дни")
ax.set_title("Среднее и медианное время в дефолте")
plt.tight_layout()
plt.show()

### 8.3. Среднее время в дефолте по тяжести дефолта

Сравниваем длительность закрытых дефолтов в зависимости от тяжести (`severity`): тяжёлый, надзор, вышел из дефолта.

In [ ]:
sev_agg = (
    df_closed.groupby("severity")["duration_days"]
    .agg(["count", "mean", "median"])
    .round(0)
    .rename(columns={"count": "Записей", "mean": "Среднее (дни)", "median": "Медиана (дни)"})
    .sort_values("Среднее (дни)", ascending=False)
)

print("Среднее время в дефолте по тяжести:")
print("=" * 55)
display(sev_agg)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(sev_agg))
w = 0.35
bars1 = ax.bar(x - w/2, sev_agg["Среднее (дни)"], w, label="Среднее",  color="tomato",    edgecolor="white")
bars2 = ax.bar(x + w/2, sev_agg["Медиана (дни)"], w, label="Медиана", color="steelblue", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(sev_agg.index, rotation=0)
ax.set_ylabel("Дни")
ax.set_title("Средняя и медианная длительность дефолта по тяжести")
ax.legend()
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f"{bar.get_height():.0f}", ha="center", fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f"{bar.get_height():.0f}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

### 8.4. Динамика и длительность дефолтов по сегментам

In [ ]:
# --- Закрытие дефолтов по кварталам в разрезе сегментов ---
closed_seg = df_def[
    df_def["end_date"].notna() &
    (df_def["end_date"] >= pd.Timestamp("2023-01-01")) &
    (df_def["end_date"] <= pd.Timestamp("2025-12-31")) &
    df_def["segment"].notna()
].copy()
closed_seg["end_quarter"] = closed_seg["end_date"].dt.to_period("Q").astype(str)

qt_seg = (
    closed_seg.groupby(["end_quarter", "segment"])["inn"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=SEG_ORDER, fill_value=0)
)

fig, ax = plt.subplots(figsize=(14, 5))
qt_seg.plot(kind="bar", ax=ax, edgecolor="white")
ax.set_xlabel("Квартал")
ax.set_ylabel("Количество компаний")
ax.set_title("Закрытие дефолтов по кварталам и сегментам")
ax.legend(title="Сегмент")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print("Закрытие дефолтов по кварталам и сегментам:")
display(qt_seg.style.format("{:,}"))

# --- Длительность дефолтов по сегментам ---
df_closed_seg = df_def[
    df_def["end_date"].notna() & df_def["segment"].notna()
].copy()
df_closed_seg["duration_days"] = (df_closed_seg["end_date"] - df_closed_seg["start_date"]).dt.days
df_closed_seg = df_closed_seg[df_closed_seg["duration_days"] >= 0]

dur_seg = (
    df_closed_seg.groupby("segment")["duration_days"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(0)
    .rename(columns={"count": "Записей", "mean": "Среднее (дни)", "median": "Медиана (дни)",
                      "min": "Мин (дни)", "max": "Макс (дни)"})
    .reindex(SEG_ORDER)
)

print("\nДлительность дефолтов по сегментам:")
display(dur_seg)

# --- Длительность по тяжести × сегмент ---
sev_seg = (
    df_closed_seg.groupby(["segment", "severity"])["duration_days"]
    .agg(["count", "mean", "median"])
    .round(0)
    .rename(columns={"count": "Записей", "mean": "Среднее (дни)", "median": "Медиана (дни)"})
)
print("\nДлительность дефолтов по сегменту и тяжести:")
display(sev_seg)

## 9. Тепловая карта корреляций между ФП

Корреляция Phi (эквивалент Пирсона для бинарных признаков) между наиболее частыми ФП. Показывает, какие ФП часто встречаются вместе.

In [ ]:
import seaborn as sns

# Строим тепловую карту только для ТОП-30 самых частых ФП,
# чтобы полная матрица не была нечитаемой
TOP_N_HEATMAP = 30
fp_freq = dataset[fp_cols].sum().sort_values(ascending=False)
top_fp = fp_freq.head(TOP_N_HEATMAP).index.tolist()

corr = dataset[top_fp].corr()
mask = np.triu(np.ones(corr.shape, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    corr, mask=mask, annot=False, cmap="RdBu_r", center=0,
    vmin=-0.5, vmax=0.5, linewidths=0.3,
    xticklabels=True, yticklabels=True, ax=ax,
)
ax.set_title(f"Корреляции между ТОП-{TOP_N_HEATMAP} наиболее частыми ФП")
plt.xticks(fontsize=7, rotation=90)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.show()

# Извлекаем верхний треугольник матрицы (без диагонали) в плоский формат
all_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
all_pairs.columns = ["ФП_1", "ФП_2", "Корреляция"]

show_cols = ["ФП_1", "ФП_2", "Корреляция"]

# Положительные корреляции (r > 0.3): ФП, которые часто встречаются вместе
strong_pos = all_pairs[all_pairs["Корреляция"] > 0.3].sort_values("Корреляция", ascending=False)
print(f"\nПары ФП с положительной корреляцией (r > 0.3): {len(strong_pos)}")
if len(strong_pos) > 0:
    display(strong_pos[show_cols].head(20).style.hide(axis="index").format({"Корреляция": "{:.3f}"}))
else:
    print("  Пар с r > 0.3 не найдено")

# Отрицательные корреляции (r < -0.3): взаимоисключающие ФП
strong_neg = all_pairs[all_pairs["Корреляция"] < -0.3].sort_values("Корреляция", ascending=True)
print(f"\nВзаимоисключающие пары ФП (r < -0.3): {len(strong_neg)}")
if len(strong_neg) > 0:
    display(strong_neg[show_cols].head(20).style.hide(axis="index").format({"Корреляция": "{:.3f}"}))
else:
    print("  Пар с r < -0.3 не найдено")

### 9.1. Тепловые карты корреляций ФП по сегментам (ККБ, МСБ, МкБ)

Строим отдельные тепловые карты корреляций ФП для трёх бизнес-сегментов:
1. **МкБ** — микробизнес
2. **МСБ** — малый и средний бизнес
3. **ККБ** — крупный корпоративный бизнес

In [ ]:
TOP_N = 30

for seg_name in SEG_ORDER:
    subset = dataset[dataset["segment"] == seg_name]
    n_companies = len(subset)
    n_defaults = int(subset["default_flag"].sum()) if n_companies > 0 else 0

    print(f"\n{'='*70}")
    if n_companies > 0:
        print(f"{seg_name}: {n_companies:,} компаний, {n_defaults:,} дефолтных "
              f"({n_defaults/n_companies:.1%} DR)")
    else:
        print(f"{seg_name}: 0 компаний")
    print("=" * 70)

    if n_companies < 10:
        print(f"  Слишком мало компаний ({n_companies}) — тепловая карта не строится")
        continue

    fp_freq_seg = subset[fp_cols].sum().sort_values(ascending=False)
    active_fp = fp_freq_seg[fp_freq_seg >= 2].head(TOP_N).index.tolist()

    if len(active_fp) < 2:
        print(f"  Менее 2 активных ФП — тепловая карта не строится")
        continue

    corr_seg = subset[active_fp].corr()
    mask_tri = np.triu(np.ones(corr_seg.shape, dtype=bool), k=1)

    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(
        corr_seg, mask=mask_tri, annot=False, cmap="RdBu_r", center=0,
        vmin=-0.5, vmax=0.5, linewidths=0.3,
        xticklabels=True, yticklabels=True, ax=ax,
    )
    ax.set_title(f"Корреляции между ТОП-{len(active_fp)} ФП — {seg_name}\n"
                 f"(компаний: {n_companies:,}, дефолтных: {n_defaults:,})")
    plt.xticks(fontsize=7, rotation=90)
    plt.yticks(fontsize=7)
    plt.tight_layout()
    plt.show()

    pairs_seg = (
        corr_seg.where(np.triu(np.ones(corr_seg.shape), k=1).astype(bool))
        .stack()
        .reset_index()
    )
    pairs_seg.columns = ["ФП_1", "ФП_2", "Корреляция"]
    strong_seg = pairs_seg[pairs_seg["Корреляция"].abs() > 0.3].sort_values(
        "Корреляция", key=abs, ascending=False
    )

    if len(strong_seg) > 0:
        print(f"  Пары с |r| > 0.3: {len(strong_seg)}")
        display(strong_seg.head(10).style.hide(axis="index").format({"Корреляция": "{:.3f}"}))
    else:
        print(f"  Пар с |r| > 0.3 не найдено")

## 10. Кросс-таблицы «ФП × Дефолт» и первичные находки по каждому типу ФП

Для каждого ФП:
- **Частота встречаемости** — у скольких компаний есть этот ФП
- **Доля дефолтов** — какой % носителей ФП оказался в дефолте
- **Lift** — во сколько раз доля дефолтов с ФП выше базовой (предварительная оценка «опасности»)

In [ ]:
# Базовый default rate — «фон», относительно которого оцениваем каждый ФП
base_rate = dataset["default_flag"].mean()
total_companies = len(dataset)
total_defaults = dataset["default_flag"].sum()

# Для каждого ФП считаем:
#   n_with         — сколько компаний имеют этот ФП
#   n_defaults_with — сколько из них дефолтных
#   rate_with      — доля дефолтов среди носителей ФП
#   lift           — во сколько раз rate_with выше базового default rate
#                    (lift > 1 = ФП «опасный», lift < 1 = ФП «безопасный»)
results = []
for col in fp_cols:
    n_with = (dataset[col] == 1).sum()
    n_defaults_with = dataset.loc[dataset[col] == 1, "default_flag"].sum()
    rate_with = n_defaults_with / n_with if n_with > 0 else 0
    lift = rate_with / base_rate if base_rate > 0 else 0

    results.append({
        "Тип ФП": col,
        "Компаний с ФП": n_with,
        "% от всех компаний": round(n_with / total_companies * 100, 2),
        "Дефолтов среди них": n_defaults_with,
        "Доля дефолтов": round(rate_with * 100, 2),
        "Lift": round(lift, 2),
    })

df_cross = pd.DataFrame(results).sort_values("Lift", ascending=False)

print(f"Базовый default rate: {base_rate:.2%}")
print(f"Всего компаний: {total_companies:,}, дефолтов: {total_defaults:,}\n")

# Показываем ТОП-30, фильтруя ФП с < 3 дефолтами (иначе lift ненадёжен)
print("="*70)
print("ТОП-30 ФП по Lift (предварительная оценка «опасности»):")
print("="*70)
display(
    df_cross[df_cross["Дефолтов среди них"] >= 3]
    .head(30)
    .style.hide(axis="index")
    .format({"Доля дефолтов": "{:.2f}%", "% от всех компаний": "{:.2f}%"})
    .bar(subset=["Lift"], color="salmon")
)

In [ ]:
print("="*70)
print(f"Полная кросс-таблица «ФП × Дефолт» (все {len(fp_cols)} ФП):")
print("="*70)
display(
    df_cross
    .style.hide(axis="index")
    .format({"Доля дефолтов": "{:.2f}%", "% от всех компаний": "{:.2f}%"})
)

### 10.2. Кросс-таблицы «ФП × Дефолт» по сегментам

In [ ]:
cross_by_seg = {}

for seg in SEG_ORDER:
    seg_data = dataset[dataset["segment"] == seg]
    if len(seg_data) == 0:
        continue

    seg_base_rate = seg_data["default_flag"].mean()
    seg_total = len(seg_data)
    seg_results = []

    for col in fp_cols:
        n_with = (seg_data[col] == 1).sum()
        if n_with == 0:
            continue
        n_def_with = seg_data.loc[seg_data[col] == 1, "default_flag"].sum()
        rate = n_def_with / n_with if n_with > 0 else 0
        lift = rate / seg_base_rate if seg_base_rate > 0 else 0
        seg_results.append({
            "Тип ФП": col,
            "Компаний с ФП": n_with,
            "% от всех": round(n_with / seg_total * 100, 2),
            "Дефолтов": n_def_with,
            "Доля дефолтов": round(rate * 100, 2),
            "Lift": round(lift, 2),
        })

    df_seg_cross = pd.DataFrame(seg_results).sort_values("Lift", ascending=False)
    cross_by_seg[seg] = df_seg_cross

    print(f"\n{'='*70}")
    print(f"{seg}: base DR = {seg_base_rate:.2%}, компаний = {seg_total:,}")
    print("="*70)
    top = df_seg_cross[df_seg_cross["Дефолтов"] >= 3].head(20)
    if len(top) > 0:
        display(
            top.style.hide(axis="index")
            .format({"Доля дефолтов": "{:.2f}%", "% от всех": "{:.2f}%"})
            .bar(subset=["Lift"], color="salmon")
        )
    else:
        print("  Нет ФП с ≥3 дефолтами")

## 10.1. Топ-5 комбинаций из 2–3 ФП, ведущих к дефолту

Перебираем пары и тройки **только** среди ФП с достаточной частотой (≥20 компаний). Для каждой комбинации считаем lift = (default rate комбинации) / (базовый default rate).

In [ ]:
from itertools import combinations

# Пороги для отсечки шумных комбинаций:
# — минимум 10 компаний с данной комбинацией ФП
# — минимум 3 дефолта среди них (иначе lift = случайность)
MIN_COMPANIES_COMBO = 10
MIN_DEFAULTS_COMBO = 3

# Отбираем только достаточно частые ФП (≥20 компаний).
# Для редких ФП комбинации почти всегда будут пустыми.
frequent_fp = [c for c in fp_cols if (dataset[c] == 1).sum() >= 20]
print(f"ФП с ≥20 компаниями (для поиска комбинаций): {len(frequent_fp)}")

# Переводим в numpy-массивы для скорости перебора
base_rate = dataset["default_flag"].mean()
y = dataset["default_flag"].values
X = dataset[frequent_fp].values
fp_names = np.array(frequent_fp)

# --- Пары (C(n,2) комбинаций) ---
# Для каждой пары: считаем компании, где оба ФП = 1, и долю дефолтов среди них
pair_results = []
for i, j in combinations(range(len(frequent_fp)), 2):
    mask = (X[:, i] == 1) & (X[:, j] == 1)
    n = mask.sum()
    if n < MIN_COMPANIES_COMBO:
        continue
    n_def = y[mask].sum()
    if n_def < MIN_DEFAULTS_COMBO:
        continue
    rate = n_def / n
    lift = rate / base_rate
    pair_results.append({
        "Комбинация": f"{fp_names[i]}  +  {fp_names[j]}",
        "Размер": 2,
        "Компаний": n,
        "Дефолтов": n_def,
        "Default rate": round(rate * 100, 2),
        "Lift": round(lift, 2),
    })

df_pairs = pd.DataFrame(pair_results).sort_values("Lift", ascending=False)
print(f"Найдено пар с ≥{MIN_COMPANIES_COMBO} компаний и ≥{MIN_DEFAULTS_COMBO} дефолтов: {len(df_pairs)}")

print(f"\n{'='*70}")
print("ТОП-5 пар ФП по Lift:")
print("="*70)
if len(df_pairs) > 0:
    display(
        df_pairs.head(5)
        .style.hide(axis="index")
        .format({"Default rate": "{:.2f}%"})
        .bar(subset=["Lift"], color="salmon")
    )
else:
    print("  Пар, удовлетворяющих порогам, не найдено.")

In [ ]:
# --- Тройки (C(n,3) комбинаций) ---
# Полный перебор троек из всех ФП — слишком долго.
# Ограничиваем: берём только ТОП-30 ФП по индивидуальному lift.
# C(30,3) = 4060 — вполне быстро.
top_individual = (
    df_cross[df_cross["Компаний с ФП"] >= 20]
    .sort_values("Lift", ascending=False)
    .head(30)["Тип ФП"]
    .tolist()
)
top_idx = [i for i, name in enumerate(frequent_fp) if name in top_individual]

triple_results = []
for i, j, k in combinations(top_idx, 3):
    mask = (X[:, i] == 1) & (X[:, j] == 1) & (X[:, k] == 1)
    n = mask.sum()
    if n < MIN_COMPANIES_COMBO:
        continue
    n_def = y[mask].sum()
    if n_def < MIN_DEFAULTS_COMBO:
        continue
    rate = n_def / n
    lift = rate / base_rate
    triple_results.append({
        "Комбинация": f"{fp_names[i]}  +  {fp_names[j]}  +  {fp_names[k]}",
        "Размер": 3,
        "Компаний": n,
        "Дефолтов": n_def,
        "Default rate": round(rate * 100, 2),
        "Lift": round(lift, 2),
    })

df_triples = pd.DataFrame(triple_results).sort_values("Lift", ascending=False) if triple_results else pd.DataFrame()
print(f"Найдено троек с ≥{MIN_COMPANIES_COMBO} компаний и ≥{MIN_DEFAULTS_COMBO} дефолтов: {len(df_triples)}")

print(f"\n{'='*70}")
print("ТОП-5 троек ФП по Lift:")
print("="*70)
if len(df_triples) > 0:
    display(
        df_triples.head(5)
        .style.hide(axis="index")
        .format({"Default rate": "{:.2f}%"})
        .bar(subset=["Lift"], color="salmon")
    )
else:
    print("  Троек, удовлетворяющих порогам, не найдено.")

# --- Итоговая сводка: объединяем пары и тройки, показываем ТОП-5 ---
df_all_combos = pd.concat([df_pairs, df_triples], ignore_index=True).sort_values("Lift", ascending=False)
print(f"\n{'='*70}")
print("ТОП-5 комбинаций (пары + тройки) по Lift:")
print("="*70)
if len(df_all_combos) > 0:
    display(
        df_all_combos.head(5)
        .style.hide(axis="index")
        .format({"Default rate": "{:.2f}%"})
        .bar(subset=["Lift"], color="salmon")
    )

### 10.3. Топ комбинаций ФП по сегментам

In [ ]:
combos_by_seg = {}

for seg in SEG_ORDER:
    seg_data = dataset[dataset["segment"] == seg]
    if len(seg_data) < 30:
        continue
    seg_base_rate = seg_data["default_flag"].mean()
    if seg_base_rate == 0:
        continue

    seg_fp = [c for c in fp_cols if (seg_data[c] == 1).sum() >= 10]
    y_seg = seg_data["default_flag"].values
    X_seg = seg_data[seg_fp].values
    fp_n = np.array(seg_fp)

    pair_res = []
    for i, j in combinations(range(len(seg_fp)), 2):
        m = (X_seg[:, i] == 1) & (X_seg[:, j] == 1)
        n = m.sum()
        if n < 5:
            continue
        nd = y_seg[m].sum()
        if nd < 2:
            continue
        r = nd / n
        pair_res.append({
            "Комбинация": f"{fp_n[i]}  +  {fp_n[j]}",
            "Размер": 2, "Компаний": n, "Дефолтов": nd,
            "Default rate": round(r * 100, 2),
            "Lift": round(r / seg_base_rate, 2),
        })

    df_p = pd.DataFrame(pair_res).sort_values("Lift", ascending=False) if pair_res else pd.DataFrame()

    top_lift = (
        cross_by_seg.get(seg, pd.DataFrame())
        .query("`Компаний с ФП` >= 10")
        .sort_values("Lift", ascending=False)
        .head(20)["Тип ФП"].tolist()
    )
    ti = [k for k, name in enumerate(seg_fp) if name in top_lift]

    triple_res = []
    for i, j, k in combinations(ti, 3):
        m = (X_seg[:, i] == 1) & (X_seg[:, j] == 1) & (X_seg[:, k] == 1)
        n = m.sum()
        if n < 5:
            continue
        nd = y_seg[m].sum()
        if nd < 2:
            continue
        r = nd / n
        triple_res.append({
            "Комбинация": f"{fp_n[i]}  +  {fp_n[j]}  +  {fp_n[k]}",
            "Размер": 3, "Компаний": n, "Дефолтов": nd,
            "Default rate": round(r * 100, 2),
            "Lift": round(r / seg_base_rate, 2),
        })

    df_t = pd.DataFrame(triple_res).sort_values("Lift", ascending=False) if triple_res else pd.DataFrame()
    df_combo = pd.concat([df_p, df_t], ignore_index=True).sort_values("Lift", ascending=False)
    combos_by_seg[seg] = df_combo

    print(f"\n{'='*70}")
    print(f"{seg}: base DR = {seg_base_rate:.2%}, пар = {len(df_p)}, троек = {len(df_t)}")
    print("="*70)
    if len(df_combo) > 0:
        display(
            df_combo.head(10)
            .style.hide(axis="index")
            .format({"Default rate": "{:.2f}%"})
            .bar(subset=["Lift"], color="salmon")
        )
    else:
        print("  Комбинаций не найдено")

## 11. Аномалии и интересные паттерны

In [ ]:
print("="*70)
print("АНОМАЛИИ И ПАТТЕРНЫ")
print("="*70)

# 1. ФП, при которых ВСЕ носители оказались дефолтными (100% default rate).
#    Порог ≥2 компании — при 1 компании 100% ничего не значит.
fp_100 = df_cross[(df_cross["Доля дефолтов"] == 100) & (df_cross["Компаний с ФП"] >= 2)]
print(f"\n1. ФП с 100% дефолтностью (≥2 компании): {len(fp_100)}")
if len(fp_100) > 0:
    display(fp_100.style.hide(axis="index"))

# 2. ФП, при которых ни один носитель не дефолтнул (0% при ≥20 компаниях).
#    Потенциально «безопасные» ФП — или просто не связанные с дефолтом.
fp_0 = df_cross[(df_cross["Доля дефолтов"] == 0) & (df_cross["Компаний с ФП"] >= 20)]
print(f"\n2. ФП с 0% дефолтностью (≥20 компаний): {len(fp_0)}")
if len(fp_0) > 0:
    display(fp_0.style.hide(axis="index"))

# 3. Редкие ФП (< 5 компаний) — по ним невозможно делать выводы,
#    стоит рассмотреть группировку в категории на следующих шагах.
rare_fp = df_cross[df_cross["Компаний с ФП"] < 5]
print(f"\n3. Редкие ФП (< 5 компаний): {len(rare_fp)} из {len(df_cross)}")

# 4. Компании-выбросы по числу ФП (> 99-го перцентиля).
#    Интересно: дефолтные они или нет?
q99 = fp_per_company.quantile(0.99)
outliers = dataset[fp_per_company > q99]
print(f"\n4. Компании с аномально большим числом ФП (> {q99:.0f}, 99-й перцентиль): {len(outliers)}")
if len(outliers) > 0:
    outlier_info = outliers[["inn", "default_flag", "severity", "fp_count"]].sort_values("fp_count", ascending=False)
    display(outlier_info.head(10).style.hide(axis="index"))

# 5. Дефолтные компании, у которых вообще нет ФП в CRM за анализируемый период.
#    Возможные причины: ФП выявлены после дефолта (отсечены временным фильтром),
#    или не заводились вовсе.
def_no_fp = dataset[(dataset["default_flag"] == 1) & (dataset["fp_count"] == 0)]
print(f"\n5. Дефолтные компании без единого ФП: {len(def_no_fp)}")
if len(def_no_fp) > 0:
    print(f"   Это {len(def_no_fp) / dataset['default_flag'].sum():.1%} от всех дефолтных")

### 11.1. Аномалии по сегментам

In [ ]:
anomalies_by_seg = {}

for seg in SEG_ORDER:
    seg_data = dataset[dataset["segment"] == seg]
    if len(seg_data) == 0:
        continue

    seg_cross = cross_by_seg.get(seg, pd.DataFrame())
    if len(seg_cross) == 0:
        continue

    print(f"\n{'='*70}")
    print(f"{seg}: {len(seg_data):,} компаний, {int(seg_data['default_flag'].sum()):,} дефолтных")
    print("="*70)

    fp100 = seg_cross[(seg_cross["Доля дефолтов"] == 100) & (seg_cross["Компаний с ФП"] >= 2)]
    print(f"\n  ФП с 100% дефолтностью (≥2 компании): {len(fp100)}")
    if len(fp100) > 0:
        display(fp100.head(10).style.hide(axis="index"))

    fp0 = seg_cross[(seg_cross["Доля дефолтов"] == 0) & (seg_cross["Компаний с ФП"] >= 10)]
    print(f"  ФП с 0% дефолтностью (≥10 компаний): {len(fp0)}")
    if len(fp0) > 0:
        display(fp0.head(10).style.hide(axis="index"))

    seg_fp_count = seg_data[fp_cols].sum(axis=1)
    q99 = seg_fp_count.quantile(0.99)
    outliers_seg = seg_data[seg_fp_count > q99]
    print(f"  Компании-выбросы (>{q99:.0f} ФП, 99-й перцентиль): {len(outliers_seg)}")

    anomalies_by_seg[seg] = {"fp_100": fp100, "fp_0": fp0, "outliers": len(outliers_seg)}

## 12. Гипотезы для углублённого анализа и рекомендации по приоритизации

In [ ]:
print("="*70)
print("ГИПОТЕЗЫ ДЛЯ УГЛУБЛЁННОГО АНАЛИЗА")
print("="*70)

n_high_lift = len(df_cross[(df_cross["Lift"] > 2) & (df_cross["Дефолтов среди них"] >= 5)])
n_corr_pairs = len(strong_pos) if len(strong_pos) > 0 else 0
n_def_no_fp = len(def_no_fp)
mean_fp_def = def_counts.mean()
mean_fp_nodef = nodef_counts.mean()

hypotheses = [
    f"Г1. Комбинации ФП: {n_corr_pairs} пар(ы) ФП с сильной корреляцией — "
    f"проверить, повышают ли они риск дефолта совместно (Apriori / lift комбинаций).",

    f"Г2. «Опасные» ФП: {n_high_lift} ФП с lift > 2 и ≥5 дефолтами — "
    f"проверить статистическую значимость (Fisher exact test, FDR-коррекция).",

    f"Г3. Количество ФП как предиктор: у дефолтных в среднем {mean_fp_def:.1f} ФП, "
    f"у недефолтных {mean_fp_nodef:.1f} — проверить, есть ли пороговое значение.",
]

if n_def_no_fp > 0:
    hypotheses.append(
        f"Г4. Дефолты без ФП: {n_def_no_fp} компаний дефолтнули без единого ФП в CRM — "
        f"возможно, ФП выявлены после дефолта или не заводились."
    )

for h in hypotheses:
    print(f"\n  {h}")

print(f"\n{'='*70}")
print("РЕКОМЕНДАЦИИ ПО ПРИОРИТИЗАЦИИ")
print("="*70)

recommendations = [
    "1. На следующем шаге (Гипотеза 1) — сфокусироваться на ФП с lift > 2 и ≥5 дефолтами, "
    "проверить их значимость и искать комбинации среди них.",

    "2. Редкие ФП (< 5 компаний) — не анализировать отдельно, "
    "рассмотреть группировку в категории.",

    "3. ФП с высокой взаимной корреляцией — учесть при моделировании "
    "(мультиколлинеарность), возможно объединить в группы.",

    "4. Проверить временную стабильность результатов: "
    "сохраняется ли рейтинг «опасности» ФП при разбиении 2022-2024 / 2025.",
]

for r in recommendations:
    print(f"\n  {r}")

## 13. Сохранение датасета и результатов EDA

In [ ]:
dataset.to_pickle(f"{DATA_DIR}/dataset_fp_default.pkl")

excel_path = f"{DATA_DIR}/приложение_build_dataset.xlsx"

def _trunc(name, limit=31):
    return name[:limit]

with pd.ExcelWriter(excel_path, engine="openpyxl") as w:
    dataset.to_excel(w, sheet_name="Датасет", index=False)

    df_cross.to_excel(w, sheet_name="Кросс-таблица (общая)", index=False)
    for seg, df_s in cross_by_seg.items():
        df_s.to_excel(w, sheet_name=_trunc(f"Кросс-таблица ({seg})"), index=False)

    if len(df_pairs) > 0:
        df_pairs.to_excel(w, sheet_name="Пары ФП (общие)", index=False)
    if len(df_triples) > 0:
        df_triples.to_excel(w, sheet_name="Тройки ФП (общие)", index=False)
    for seg, df_c in combos_by_seg.items():
        if len(df_c) > 0:
            df_c.to_excel(w, sheet_name=_trunc(f"Комбинации ({seg})"), index=False)

    fp_100.to_excel(w, sheet_name="100% дефолтность", index=False)
    fp_0.to_excel(w, sheet_name="0% дефолтность", index=False)
    rare_fp.to_excel(w, sheet_name="Редкие ФП", index=False)
    if len(outliers) > 0:
        outlier_info.to_excel(w, sheet_name="Компании-выбросы", index=False)

    qt_seg.to_excel(w, sheet_name="Дефолты по кварталам (сегм.)")
    dur_seg.to_excel(w, sheet_name="Длительность (сегменты)")
    sev_seg.to_excel(w, sheet_name="Длительность (тяж.×сегм.)")

    pd.DataFrame(seg_stats).to_excel(w, sheet_name="Статистика ФП по сегм.", index=False)

print(f"Сохранено: {excel_path}")

## 14. Генерация Word-отчёта (report_build_dataset.docx)

In [ ]:
from docx import Document
from docx.shared import Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import nsdecls, qn
from docx.oxml import parse_xml, OxmlElement

REPORT_PATH = f"{DATA_DIR}/report_build_dataset.docx"
HDR_CLR = "1F4E79"
ALT_CLR = "D6E4F0"


def _shade(cell, color):
    cell._tc.get_or_add_tcPr().append(
        parse_xml(f'<w:shd {nsdecls("w")} w:fill="{color}"/>'))


def _cw(cell, text, sz=9, bold=False, align=None, white=False):
    cell.text = ""
    p = cell.paragraphs[0]
    if align:
        p.alignment = align
    r = p.add_run(str(text))
    r.font.size = Pt(sz)
    r.font.bold = bold
    r.font.name = "Calibri"
    r.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF) if white else RGBColor(0, 0, 0)


def _set_col_widths(table, widths_cm):
    tbl = table._tbl
    tblPr = tbl.tblPr if tbl.tblPr is not None else OxmlElement("w:tblPr")
    layout = OxmlElement("w:tblLayout")
    layout.set(qn("w:type"), "fixed")
    tblPr.append(layout)
    grid = tbl.find(qn("w:tblGrid"))
    if grid is None:
        grid = OxmlElement("w:tblGrid")
        tbl.insert(1, grid)
    else:
        for child in list(grid):
            grid.remove(child)
    for w in widths_cm:
        gc = OxmlElement("w:gridCol")
        gc.set(qn("w:w"), str(int(w * 567)))
        grid.append(gc)
    for row in table.rows:
        for i, w in enumerate(widths_cm):
            if i < len(row.cells):
                tc = row.cells[i]._tc
                tcPr = tc.get_or_add_tcPr()
                tw = OxmlElement("w:tcW")
                tw.set(qn("w:w"), str(int(w * 567)))
                tw.set(qn("w:type"), "dxa")
                existing = tcPr.find(qn("w:tcW"))
                if existing is not None:
                    tcPr.remove(existing)
                tcPr.insert(0, tw)


def _mktable(doc, headers, rows, font_sz=9, num_cols=None, col_widths=None):
    t = doc.add_table(rows=1 + len(rows), cols=len(headers))
    t.style = "Table Grid"
    t.alignment = WD_TABLE_ALIGNMENT.CENTER
    if num_cols is None:
        num_cols = set()
    for i, h in enumerate(headers):
        _cw(t.rows[0].cells[i], h, font_sz, bold=True,
            align=WD_ALIGN_PARAGRAPH.CENTER, white=True)
        _shade(t.rows[0].cells[i], HDR_CLR)
    for ri, row in enumerate(rows):
        for ci, val in enumerate(row):
            al = WD_ALIGN_PARAGRAPH.RIGHT if ci in num_cols else None
            _cw(t.rows[ri + 1].cells[ci], val, font_sz, align=al)
        if ri % 2 == 1:
            for ci in range(len(headers)):
                _shade(t.rows[ri + 1].cells[ci], ALT_CLR)
    if col_widths:
        _set_col_widths(t, col_widths)
    doc.add_paragraph()
    return t


def _add_toc(doc):
    p = doc.add_paragraph()
    r = p.add_run()
    fc = OxmlElement("w:fldChar"); fc.set(qn("w:fldCharType"), "begin"); r._r.append(fc)
    r2 = p.add_run()
    ins = OxmlElement("w:instrText"); ins.set(qn("xml:space"), "preserve")
    ins.text = ' TOC \\o "1-2" \\h \\z \\u '; r2._r.append(ins)
    r3 = p.add_run()
    fs = OxmlElement("w:fldChar"); fs.set(qn("w:fldCharType"), "separate"); r3._r.append(fs)
    r4 = p.add_run("(Оглавление — нажмите Ctrl+A, затем F9 для обновления)")
    r4.font.size = Pt(10); r4.font.italic = True; r4.font.color.rgb = RGBColor(0x80, 0x80, 0x80)
    r5 = p.add_run()
    fe = OxmlElement("w:fldChar"); fe.set(qn("w:fldCharType"), "end"); r5._r.append(fe)


def _heading(doc, text, level=2):
    h = doc.add_heading(text, level=level)
    for run in h.runs:
        run.font.color.rgb = RGBColor(0, 0, 0)


def _para(doc, text):
    p = doc.add_paragraph(text)
    for run in p.runs:
        run.font.color.rgb = RGBColor(0, 0, 0)
    return p


def _conclusion(doc, text):
    p = doc.add_paragraph()
    r = p.add_run("Вывод: ")
    r.font.size = Pt(10); r.font.bold = True; r.font.italic = True
    r.font.name = "Calibri"; r.font.color.rgb = RGBColor(0, 0, 0)
    r = p.add_run(text)
    r.font.size = Pt(10); r.font.italic = True
    r.font.name = "Calibri"; r.font.color.rgb = RGBColor(0, 0, 0)


def _fmt(n):
    return f"{n:,}".replace(",", " ")


print("Утилиты отчёта загружены.")

In [ ]:
doc = Document()
for sec in doc.sections:
    sec.top_margin = Cm(1); sec.bottom_margin = Cm(1)
    sec.left_margin = Cm(1.5); sec.right_margin = Cm(1)

style = doc.styles["Normal"]
style.font.name = "Calibri"; style.font.size = Pt(10)
style.font.color.rgb = RGBColor(0, 0, 0)
for hs_name in ["Heading 1", "Heading 2"]:
    if hs_name in doc.styles:
        doc.styles[hs_name].font.color.rgb = RGBColor(0, 0, 0)

# ── Титул ──
p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
r = p.add_run("Отчёт по EDA:\nсборка аналитического датасета ФП → Дефолт")
r.font.size = Pt(20); r.font.bold = True; r.font.color.rgb = RGBColor(0, 0, 0)

p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
r = p.add_run(f"Период анализа: {date_from.strftime('%Y-%m')} — {date_to.strftime('%Y-%m')}")
r.font.size = Pt(12); r.font.color.rgb = RGBColor(0x80, 0x80, 0x80)

doc.add_page_break()
_add_toc(doc)
doc.add_page_break()

# ── Введение ──
_heading(doc, "Введение", level=1)
_para(doc,
    "Настоящий отчёт содержит результаты разведочного анализа данных (EDA) "
    "по факторам проблемности (ФП) и связи между ними и дефолтами компаний. "
    "Единица наблюдения — компания (ИНН). Итоговый датасет формируется из трёх "
    "источников: CRM (факторы проблемности), H2.0 (сегмент), данные о дефолтах.")
_para(doc,
    "Анализ выполнен в разрезе трёх бизнес-сегментов: "
    "МкБ (микробизнес), МСБ (малый и средний бизнес), ККБ (крупный корпоративный бизнес). "
    "Полные таблицы приведены в Excel-приложении «приложение_build_dataset.xlsx».")

# ═══════════════════════════════════════════════════════════════
# 1. Анализ исходных данных
# ═══════════════════════════════════════════════════════════════
_heading(doc, "1. Анализ исходных данных", level=1)
_para(doc, "Для формирования датасета использованы три источника данных:")
_mktable(doc,
    ["Источник", "Роль", "Строк", "Колонок"],
    [
        ("CRM", "Основной (ФП + реестр)", _fmt(len(df_crm)), str(df_crm.shape[1])),
        ("H2.0", "Вспомогательный (сегмент)", _fmt(len(df_h2o)), str(df_h2o.shape[1])),
        ("Дефолты", "Целевая переменная", _fmt(len(df_def)), str(df_def.shape[1])),
    ],
    num_cols={2, 3}, col_widths=[3.5, 5, 3, 2])

_para(doc,
    "Из CRM отобраны записи по трём источникам ФП: Н2.0, Справочник CRM-системы, "
    "CRM-система. Временной фильтр: ФП учитываются только если выявлены строго до "
    "даты дефолта (для недефолтных — до конца периода). Это исключает data leakage.")

_heading(doc, "1.1. Классификация тяжести дефолтов", level=2)
sev_vc = defaults["severity"].value_counts()
sev_rows = [(s, _fmt(int(sev_vc.get(s, 0)))) for s in
            ["тяжёлый", "активный", "надзор", "вышел из дефолта"]]
_mktable(doc, ["Тяжесть", "Количество"], sev_rows,
         num_cols={1}, col_widths=[6, 4])

_conclusion(doc,
    f"Всего {_fmt(len(defaults))} уникальных компаний-дефолтников за период. "
    "Классификация позволяет разделить дефолты на необратимые (тяжёлые) "
    "и потенциально обратимые.")

# ═══════════════════════════════════════════════════════════════
# 2. Итоговые данные для анализа
# ═══════════════════════════════════════════════════════════════
_heading(doc, "2. Итоговые данные для анализа", level=1)

n_total = len(dataset)
n_def = int(dataset["default_flag"].sum())
n_fp_cols = len(fp_cols)

_para(doc,
    f"Сформирован итоговый датасет: {_fmt(n_total)} компаний × {n_fp_cols + 4} колонок "
    f"({n_fp_cols} бинарных ФП-признаков + метаданные), из которых {_fmt(n_def)} дефолтных "
    f"(default rate {n_def/n_total:.2%}).")

seg_rows = []
for seg in SEG_ORDER:
    sub = dataset[dataset["segment"] == seg]
    nd = int(sub["default_flag"].sum())
    seg_rows.append((seg, _fmt(len(sub)), _fmt(nd),
                     f"{nd/len(sub):.2%}" if len(sub) > 0 else "—"))
no_seg = int(dataset["segment"].isna().sum())
if no_seg > 0:
    seg_rows.append(("Без сегмента", _fmt(no_seg), "—", "—"))

_mktable(doc, ["Сегмент", "Компаний", "Дефолтных", "Default rate"],
         seg_rows, num_cols={1, 2}, col_widths=[3.5, 3.5, 3.5, 3.5])

sudden = n_def_in_companies - n_def_in_dataset
if sudden > 0:
    _para(doc,
        f"«Внезапные дефолты» — {_fmt(sudden)} компаний дефолтнули без единого ФП, "
        "выявленного до даты дефолта (ФП появились позже или не заводились в CRM).")

_conclusion(doc,
    "Датасет сформирован корректно: временной фильтр исключает ФП, появившиеся "
    "после дефолта. Данные разбиты по трём бизнес-сегментам.")

# ═══════════════════════════════════════════════════════════════
# 3. Распределение ФП: дефолтные vs недефолтные
# ═══════════════════════════════════════════════════════════════
_heading(doc, "3. Распределение ФП: дефолтные vs недефолтные", level=1)

_para(doc,
    "Среднее, медианное и максимальное количество уникальных ФП на одну компанию, "
    "отдельно для дефолтных и недефолтных, по сегментам:")

fp_stat_rows = []
for seg in SEG_ORDER:
    sub = dataset[dataset["segment"] == seg]
    if len(sub) == 0:
        continue
    fpc = sub[fp_cols].sum(axis=1)
    d = fpc[sub["default_flag"] == 1]
    nd_ = fpc[sub["default_flag"] == 0]
    fp_stat_rows.append((
        seg,
        f"{d.mean():.1f}" if len(d) > 0 else "—",
        f"{nd_.mean():.1f}" if len(nd_) > 0 else "—",
        str(int(d.median())) if len(d) > 0 else "—",
        str(int(nd_.median())) if len(nd_) > 0 else "—",
    ))

_mktable(doc,
    ["Сегмент", "Ср. ФП (деф.)", "Ср. ФП (недеф.)", "Мед. (деф.)", "Мед. (недеф.)"],
    fp_stat_rows, num_cols={1, 2, 3, 4}, col_widths=[2.5, 3, 3, 3, 3])

_conclusion(doc,
    "У дефолтных компаний в среднем больше факторов проблемности. "
    "Количество ФП может служить индикатором риска дефолта.")

# ═══════════════════════════════════════════════════════════════
# 4. Динамика и длительность дефолтов
# ═══════════════════════════════════════════════════════════════
_heading(doc, "4. Динамика и длительность дефолтов", level=1)

_para(doc,
    "Длительность дефолта = дата окончания − дата начала. "
    "Анализ по сегментам:")

dur_rows = []
for seg in SEG_ORDER:
    row = dur_seg.loc[seg] if seg in dur_seg.index else None
    if row is not None:
        dur_rows.append((
            seg, _fmt(int(row["Записей"])),
            str(int(row["Среднее (дни)"])), str(int(row["Медиана (дни)"])),
            str(int(row["Мин (дни)"])), str(int(row["Макс (дни)"])),
        ))

_mktable(doc,
    ["Сегмент", "Записей", "Среднее (дни)", "Медиана (дни)", "Мин", "Макс"],
    dur_rows, num_cols={1, 2, 3, 4, 5}, col_widths=[2.5, 2.5, 3, 3, 2, 2.5])

_conclusion(doc,
    "Длительность дефолтов различается по сегментам. "
    "Детальная разбивка по тяжести и кварталам — в Excel-приложении.")

# ═══════════════════════════════════════════════════════════════
# 5. Корреляция ФП
# ═══════════════════════════════════════════════════════════════
_heading(doc, "5. Влияние ФП на вероятность дефолта (корреляция)", level=1)

_para(doc,
    "Корреляция Phi (эквивалент Пирсона для бинарных признаков) между ТОП-30 "
    "наиболее частых ФП, отдельно для каждого сегмента. "
    "Ориентиры: |r| > 0.3 — умеренная, |r| > 0.5 — сильная.")

n_pos = len(strong_pos) if len(strong_pos) > 0 else 0
n_neg = len(strong_neg) if len(strong_neg) > 0 else 0
_para(doc,
    f"По всем сегментам обнаружено {n_pos} пар(ы) с положительной корреляцией "
    f"(r > 0.3) и {n_neg} пар(ы) с отрицательной (r < −0.3).")

_conclusion(doc,
    "Пары с высокой корреляцией — кандидаты для группировки при моделировании. "
    "Подробные тепловые карты по сегментам — в Excel-приложении.")

# ═══════════════════════════════════════════════════════════════
# 6. Кросс-таблицы ФП по сегментам
# ═══════════════════════════════════════════════════════════════
_heading(doc, "6. Кросс-таблицы ФП по сегментам", level=1)

_para(doc,
    "Для каждого ФП: частота, доля дефолтов среди носителей, Lift "
    "(во сколько раз доля выше базовой). Lift > 1 — повышает риск.")

TOP_N_REPORT = 10

_heading(doc, "6.1. ТОП-10 ФП по Lift (все сегменты)", level=2)
top_all = df_cross[df_cross["Дефолтов среди них"] >= 3].head(TOP_N_REPORT)
_mktable(doc,
    ["Тип ФП", "Компаний", "Дефолтов", "Доля деф. %", "Lift"],
    [(r["Тип ФП"], _fmt(r["Компаний с ФП"]), _fmt(r["Дефолтов среди них"]),
      f"{r['Доля дефолтов']:.1f}%", f"{r['Lift']:.2f}")
     for _, r in top_all.iterrows()],
    num_cols={1, 2, 3, 4}, col_widths=[5, 2.5, 2.5, 2.5, 2])

for seg in SEG_ORDER:
    seg_cr = cross_by_seg.get(seg, pd.DataFrame())
    if len(seg_cr) == 0:
        continue
    top_s = seg_cr[seg_cr["Дефолтов"] >= 2].head(TOP_N_REPORT)
    if len(top_s) == 0:
        continue
    _heading(doc, f"6.2. ТОП-10 ФП по Lift — {seg}", level=2)
    _mktable(doc,
        ["Тип ФП", "Компаний", "Дефолтов", "Доля деф. %", "Lift"],
        [(r["Тип ФП"], _fmt(r["Компаний с ФП"]), _fmt(r["Дефолтов"]),
          f"{r['Доля дефолтов']:.1f}%", f"{r['Lift']:.2f}")
         for _, r in top_s.iterrows()],
        num_cols={1, 2, 3, 4}, col_widths=[5, 2.5, 2.5, 2.5, 2])

_conclusion(doc,
    "ФП с Lift > 2 и достаточным количеством наблюдений заслуживают "
    "отдельного внимания при моделировании. Полные таблицы — в Excel-приложении.")

# ═══════════════════════════════════════════════════════════════
# 7. Комбинации ФП по сегментам
# ═══════════════════════════════════════════════════════════════
_heading(doc, "7. Комбинации ФП по сегментам", level=1)

_para(doc,
    "Пары и тройки ФП. Для каждой комбинации: количество компаний, "
    "дефолтов, default rate и Lift.")

if len(df_all_combos) > 0:
    _heading(doc, "7.1. ТОП-5 комбинаций (все сегменты)", level=2)
    _mktable(doc,
        ["Комбинация", "Разм.", "Комп.", "Деф.", "DR %", "Lift"],
        [(r["Комбинация"], str(r["Размер"]), _fmt(r["Компаний"]),
          _fmt(r["Дефолтов"]), f"{r['Default rate']:.1f}%", f"{r['Lift']:.2f}")
         for _, r in df_all_combos.head(5).iterrows()],
        num_cols={1, 2, 3, 4, 5}, col_widths=[6, 1.2, 1.8, 1.8, 1.8, 1.8])

for seg in SEG_ORDER:
    df_c = combos_by_seg.get(seg, pd.DataFrame())
    if len(df_c) == 0:
        continue
    _heading(doc, f"7.2. ТОП-5 комбинаций — {seg}", level=2)
    _mktable(doc,
        ["Комбинация", "Разм.", "Комп.", "Деф.", "DR %", "Lift"],
        [(r["Комбинация"], str(r["Размер"]), _fmt(r["Компаний"]),
          _fmt(r["Дефолтов"]), f"{r['Default rate']:.1f}%", f"{r['Lift']:.2f}")
         for _, r in df_c.head(5).iterrows()],
        num_cols={1, 2, 3, 4, 5}, col_widths=[6, 1.2, 1.8, 1.8, 1.8, 1.8])

_conclusion(doc,
    "Комбинации ФП часто имеют более высокий Lift, чем отдельные ФП. "
    "Сочетание нескольких факторов значительно повышает вероятность дефолта.")

# ═══════════════════════════════════════════════════════════════
# 8. Аномалии
# ═══════════════════════════════════════════════════════════════
_heading(doc, "8. Аномалии и паттерны", level=1)

_para(doc,
    f"1. ФП с 100% дефолтностью (≥2 компании): {len(fp_100)}. "
    "Все носители оказались дефолтными.")
_para(doc,
    f"2. ФП с 0% дефолтностью (≥20 компаний): {len(fp_0)}. "
    "Ни один носитель не дефолтнул.")
_para(doc,
    f"3. Редкие ФП (< 5 компаний): {len(rare_fp)}. "
    "Статистически незначимы, рекомендуется группировка.")

_conclusion(doc,
    "«Опасность» ФП существенно различается по сегментам. "
    "Детальные таблицы аномалий по каждому сегменту — в Excel-приложении.")

# ═══════════════════════════════════════════════════════════════
# Итоговый вывод
# ═══════════════════════════════════════════════════════════════
_heading(doc, "Итоговый вывод", level=1)

_para(doc,
    "Разведочный анализ подтвердил ключевые гипотезы:")
_para(doc,
    "1. Количество ФП положительно связано с вероятностью дефолта: "
    "дефолтные компании имеют в среднем больше ФП.")
_para(doc,
    "2. Отдельные ФП имеют Lift значительно выше 1, что указывает "
    "на их прогностическую ценность при оценке кредитного риска.")
_para(doc,
    "3. Комбинации из 2–3 ФП демонстрируют кумулятивный эффект — "
    "Lift комбинации превышает Lift каждого отдельного ФП.")
_para(doc,
    "4. Паттерны различаются по сегментам: ФП, значимые для ККБ, "
    "могут не быть значимыми для МкБ. Это подтверждает необходимость "
    "сегментного подхода к моделированию.")
_para(doc,
    "Полные данные — в Excel-приложении «приложение_build_dataset.xlsx».")

doc.save(REPORT_PATH)
print(f"Отчёт сохранён: {REPORT_PATH}")